# Visualizing time series data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/06_learning_with_time_series/01a-times_series_visualization.ipynb)

```
Authors: Thomas Moreau
         Pedro Rodrigues
         Alexandre Gramfort
```

Licensed under [CC BY 4.0 Creative Commons](http://creativecommons.org/licenses/by/4.0/)

Before modelling a time series, look at it. This notebook goes through the plots
that reveal its structure:

1. **Trend and seasonality** on a small monthly series, the number of airline passengers from 1949 to 1960.
2. **Decomposition** of the series into trend, seasonality and residuals.
3. **Dependence on the past** with lag plots and autocorrelation functions.
4. **High-frequency data**: hourly bike counts in Paris, with resampling, rolling windows, a calendar heatmap and a periodogram.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

URL_REPO = "https://github.com/x-datascience-datacamp/datacamp-master/raw/main/06_learning_with_time_series/"


def data_path(name):
    """Local file if the repo is cloned, its GitHub URL otherwise (Colab)."""
    path = Path("data") / name
    return path if path.exists() else f"{URL_REPO}data/{name}"

We parse the dates while reading the file and use them as the index. Setting the
frequency (`MS`: month start) tells pandas and statsmodels that the series is
regularly sampled.

In [ ]:
df = pd.read_csv(
    data_path("AirPassengers.csv"), parse_dates=["Month"], index_col="Month"
).asfreq("MS")
df

## Line plots

In [ ]:
df.loc[:"1950"].plot();

In [ ]:
ax = sns.lineplot(data=df, x=df.index, y="Passengers")
ax.set_title("Airline passengers over time");

## Multiplicative seasonality and the log transform

The yearly oscillation grows with the number of passengers: the seasonality is
**multiplicative**. Taking the log turns it into an additive one, with a
constant amplitude. Many models (linear models, ARIMA, the decomposition below)
assume additive components, so the log is often the first transform to try.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
df["Passengers"].plot(ax=axes[0], title="Passengers")
np.log(df["Passengers"]).plot(ax=axes[1], title="log(Passengers)")
plt.tight_layout()

## Seasonal plot

In [ ]:
df["Year"] = df.index.year
df["Month"] = df.index.strftime("%b")

ax = sns.lineplot(data=df, x="Month", y="Passengers", hue="Year", legend="full")
ax.set_title("Seasonal plot")
ax.legend(bbox_to_anchor=(1.05, 1), loc=2);

## Month plot (and quarter plot)

The month plot groups the values by month: each mini-series shows how one month
evolves over the years, and the horizontal line is its mean.

In [ ]:
from statsmodels.graphics.tsaplots import month_plot, quarter_plot

month_plot(df["Passengers"], ylabel="Passengers");

In [ ]:
quarter_plot(df["Passengers"].resample("QE").sum(), ylabel="Passengers");

## Polar seasonal plot

Wrapping each year around a circle shows the seasonal pattern and the trend
together. The angle increases continuously across years, so December connects
to the next January.

In [ ]:
s = df["Passengers"]
theta = 2 * np.pi * ((s.index.year - s.index.year[0]) + (s.index.month - 1) / 12)

fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(6, 6))
cmap = plt.get_cmap("viridis", s.index.year.nunique())
for i in range(0, len(s), 12):
    # one year plus the next January, to close the loop
    ax.plot(theta[i:i + 13], s.iloc[i:i + 13], color=cmap(i // 12), label=s.index.year[i])
ax.set_xticks(2 * np.pi * np.arange(12) / 12)
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_title("Passengers over time")
ax.legend(bbox_to_anchor=(1.1, 1), loc=2);

## Decomposition: trend + seasonality + residuals

STL (*Seasonal-Trend decomposition using LOESS*) splits a series into a smooth
trend, a seasonal component of a given period, and residuals. We apply it to the
log series, where the seasonality is additive. With `robust=True`, outliers end
up in the residuals instead of distorting the trend.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(np.log(df["Passengers"]), period=12, robust=True).fit()
with plt.rc_context({"figure.figsize": (8, 6)}):
    stl.plot();

The residuals look like noise: the trend and a fixed yearly pattern explain
almost everything. A forecasting model for this series mostly needs to
extrapolate the trend and repeat the seasonality.

## Lag plots

A lag plot shows each value against the value $k$ steps earlier. Points on the
diagonal mean the series at time $t$ is well predicted by its value at $t-k$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for lag, ax in zip([1, 6, 12], axes):
    pd.plotting.lag_plot(df["Passengers"], lag=lag, ax=ax)
    ax.set_title(f"Lag {lag}")
    ax.set_xlabel("Passengers at month $t$")
    ax.set_ylabel(f"Passengers at month $t + {lag}$")
plt.tight_layout()

Lag 1 and lag 12 are close to the diagonal (same season one year apart), while
lag 6 opposes summer and winter months and is more spread.

## Autocorrelation plots

The autocorrelation function (ACF) is the correlation of the series with a
delayed copy of itself, as a function of the delay (lag): it summarizes all the
lag plots in one curve.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ax = plot_acf(df["Passengers"]).axes[0]
ax.grid(True)

In the ACF plot, we can see that there are **significant autocorrelations** (above the 95% confidence interval, corresponding to the default 5% significance level).

The slow decrease in the ACF as the lags increase is due to the trend, while the "scalloped" shape is due the seasonality.

Let's now look at the **partial autocorrelation function (PACF)**.

In [ ]:
ax = plot_pacf(df["Passengers"]).axes[0]
ax.grid(True)

PACF looks at the correlation between the residuals (the remainder after removing the effects explained by the previous lags) and the following lag value. This way, we effectively remove the already found variations before we find the next correlation. In practice, a high partial correlation indicates that there is **some information in the residual that can be modeled by the next lag**. So we **might consider keeping that lag as a feature in our model**.

Let's see how these plots would look like for white noise:

In [ ]:
rng = np.random.default_rng(42)
white_noise = rng.normal(size=300)

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_acf(white_noise, ax=axes[0])
plot_pacf(white_noise, ax=axes[1])
plt.tight_layout()

<div class="alert alert-warning">

**Careful:** Non-negative partial autocorrelations are not to be interpreted as
a sign of the frequency of the signal. They are a measure of which lags are
useful to predict the signal.

</div>

In [ ]:
sine = pd.Series(np.sin(2 * np.pi / 12 * df.index.month), index=df.index)

fig, axes = plt.subplots(3, 1, figsize=(6, 6))
sine.plot(ax=axes[0])
plot_acf(sine, ax=axes[1])
plot_pacf(sine, ax=axes[2])
plt.tight_layout()

<div class="alert alert-success">

**EXERCISE:**

- STL estimated the trend with local regressions. Implement a simpler way to remove the trend (for instance a linear fit or a moving average), then plot the autocorrelation and partial autocorrelation functions of the detrended data.

</div>

Solution is in: `solutions/visu-1_ts-detrending.py`

In [ ]:
from sklearn.linear_model import LinearRegression

trend = ...

residual = df["Passengers"] - trend

fig, axes = plt.subplots(4, 1, figsize=(8, 6))
axes[0].plot(df["Passengers"])
axes[0].plot(trend, "r")
axes[1].plot(residual)
axes[1].set_title("Residuals")
plot_acf(residual, ax=axes[2])
plot_pacf(residual, ax=axes[3])
plt.tight_layout()

# High-frequency data: hourly bike counts

The second series is the hourly number of bikes counted on boulevard de
Sébastopol in Paris, from September 2020 to October 2021. It is the dataset of
the forecasting case study (`02c`). With almost 10,000 points, plotting it raw
shows little, and we need tools to look at it at several time scales.

In [ ]:
bikes = pd.read_parquet(data_path("bike-counter-data.parquet"))
counts = (
    bikes.query("counter_name == 'Totem 73 boulevard de Sébastopol S-N'")
    .set_index("date")["bike_count"]
    .sort_index()
)

### Check the sampling

Most models assume a regular grid. The differences between consecutive
timestamps show whether some are missing.

In [ ]:
counts.index.to_series().diff().value_counts()

One step is 2 hours long. `asfreq` puts the series on a regular hourly grid and
inserts `NaN` where a timestamp is missing:

In [ ]:
counts = counts.asfreq("h")
counts[counts.isna()]

The missing hour is in the night of October 25th, 2020, when France switched
from summer to winter time: the timestamps are in local time. We fill it by
interpolation.

In [ ]:
counts = counts.interpolate()

### Resampling and rolling windows

`resample` aggregates the series on a coarser grid (here daily totals), and
`rolling` computes statistics on a sliding window (here a 7-day mean, which
removes the weekly pattern).

In [ ]:
daily = counts.resample("D").sum()

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
counts.plot(ax=axes[0], lw=0.3, title="Hourly counts")
daily.plot(ax=axes[1], alpha=0.5, label="daily total")
daily.rolling(7, center=True).mean().plot(ax=axes[1], label="7-day rolling mean")
axes[1].legend()
plt.tight_layout()

<div class="alert alert-warning">

**Careful:** `center=True` averages over past *and future* days. It is fine to
look at the data, but such a feature would leak the future into a forecasting
model. See the notebook on evaluation (`01b`).

</div>

### Calendar heatmap

Arranging the daily totals by week (columns) and day of the week (rows) shows
weekly patterns, holidays and unusual periods at a glance.

In [ ]:
calendar = pd.DataFrame({
    "count": daily,
    "week": daily.index.to_period("W").start_time.date,
    "weekday": daily.index.day_name().str[:3],
}).pivot(index="weekday", columns="week", values="count")
calendar = calendar.loc[["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]]

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(calendar, cmap="viridis", ax=ax, xticklabels=4, yticklabels=True,
            cbar_kws={"label": "bikes per day"});

Weekends are lighter than weekdays. The dark weeks in November 2020 are the
second COVID lockdown, followed by the Christmas holidays, and the summer
holidays empty the city in August. The data alone does not explain them: external information
(holidays, lockdowns, weather) is needed to model them.

### Periodogram

The periodogram measures how much of the variance comes from each frequency.
Peaks reveal the periods of the seasonal patterns, even when there are several
of them.

In [ ]:
from scipy.signal import periodogram

freqs, power = periodogram(np.log1p(counts), fs=24)  # fs: samples per day

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(1 / freqs[1:], power[1:])
for period in [0.5, 1, 7]:
    ax.axvline(period, color="k", ls="--", lw=0.8)
ax.set_xlabel("Period (days)")
ax.set_ylabel("Power");

The main peaks are at 1 day and 7 days (the daily and weekly cycles), plus
12 hours, a harmonic of the daily cycle with its two commute peaks. A model of
this series needs features for at least the hour of the day and the day of the
week.